# Lecture 23: Data Visualization & the Grammar of Graphics

**Ling 250/450: Data Science for Linguistics — Spring 2026**

Almost every plot you will ever need to make in R can be built with one package — `ggplot2` — and one idea: that a visualization is not a *type of chart* you pick from a menu, but a **composition of independent components** you assemble. That idea is called the **Grammar of Graphics**, and it is the main thing this lecture is about.

The word *grammar* is deliberate. Just as English grammar lets you generate an unbounded number of sentences from a small set of parts, the grammar of graphics lets you generate an unbounded number of plots from a small set of parts. Once you know the parts, you can build almost any plot you can imagine — and, just as importantly, you can *critique* any plot someone else makes by asking which parts they chose and whether those choices were good ones.

### Why this lecture deemphasizes syntax

In previous years, this lecture spent a lot of time on the specific function names in `ggplot2` — `geom_point`, `scale_x_reverse`, `facet_wrap`, and so on. That is exactly the kind of thing an LLM is very good at producing for you on demand. What the LLM *cannot* do for you is decide **which component you need** for the question you are trying to answer, or tell you that your choice of color channel for an ordinal variable is going to mislead your reader. That decision-making is conceptual, and it is what we want to practice today.

### Reading references

- Wilkinson, L. (1999/2005). *The Grammar of Graphics.* Springer. [[link]](https://link.springer.com/book/10.1007/0-387-28695-0) (The book that introduced the idea.)
- Wickham, H. (2010). "A Layered Grammar of Graphics." *Journal of Computational and Graphical Statistics*, 19(1). [[pdf]](http://vita.had.co.nz/papers/layered-grammar.pdf) (The paper that turned it into `ggplot2`.)
- Cleveland, W. S. & McGill, R. (1984). "Graphical Perception: Theory, Experimentation, and Application to the Development of Graphical Methods." *JASA*, 79(387). [[link]](http://euclid.psych.yorku.ca/www/psy6135/papers/ClevelandMcGill1984.pdf) (Where the ranking of visual channels comes from.)

## The components of a plot

Wickham's "layered" version of the grammar breaks a plot into seven components. We will introduce each one conceptually, then show the `ggplot2` syntax that realizes it.

| # | Component | The question it answers |
|---|-----------|-------------------------|
| 1 | **Data** | What rows and columns am I drawing from? |
| 2 | **Aesthetic mappings** | Which *variables* get assigned to which *visual channels* (x, y, color, size, shape)? |
| 3 | **Geometries** | What visual marks do I draw — points, lines, bars, boxes? |
| 4 | **Statistics** | Do I draw the raw values, or a transformation of them (a bin count, a smoother, a summary)? |
| 5 | **Scales** | How do data values become visual values — what axis range, what colors, what transformations? |
| 6 | **Coordinates** | What coordinate system — Cartesian, polar, flipped? |
| 7 | **Facets** | Do I repeat the same plot across subsets of the data? |

The critical conceptual move is #2: a variable in your data frame is **not** "the x-axis." It is a variable that you *map* to the x channel — and could just as well map to color, or size, or a facet. The same data supports many plots because the mappings can change.

## Setup

We only need two libraries today. `ggplot2` is the plotting library; `dplyr` we have seen before for data manipulation.

In [ ]:
library(ggplot2)
library(dplyr)

# Ensure UTF-8 locale so non-ASCII characters (e.g. IPA) display correctly
Sys.setlocale("LC_ALL", "en_US.UTF-8")

## Component 1: Data

Every ggplot starts with a data frame. `ggplot2` expects data in **tidy** form: one row per observation, one column per variable. This is not just a convention — the entire grammar assumes it, because aesthetic mappings are stated in terms of *column names*.

Let us build a small synthetic dataset to work with. Imagine we are studying algae growth in two ponds as a function of fertilizer concentration. We will also record water temperature as a secondary variable.

In [ ]:
set.seed(1)

fertilizer <- runif(n = 200, max = 100)
pond <- c(rep("pond_1", 100), rep("pond_2", 100))

# Algae grows roughly linearly with fertilizer, but the two ponds have
# different slopes and intercepts.
algae_1 <- 1.0 * fertilizer[1:100] + 10 + rnorm(100, sd = 10)
algae_2 <- 0.5 * fertilizer[101:200] + 40 + rnorm(100, sd = 10)
algae <- c(algae_1, algae_2)

temperature <- runif(200, max = 100)

algae_data <- data.frame(fertilizer, pond, algae, temperature)
head(algae_data)

Notice the shape: four columns, 200 rows, one observation per row. Each column is a *variable* that we will be free to map to any visual channel we want. `fertilizer` is not inherently the x-axis — it is just a column, and we will *choose* to put it on x.

## Component 2: Aesthetic mappings

An **aesthetic mapping** is a statement of the form: "the value of variable *V* will be represented by visual channel *C*." Common visual channels are:

- **position** (x, y) — by far the most informative channel
- **color** (hue, or fill for filled shapes)
- **size**
- **shape**
- **alpha** (transparency)
- **linetype**

In ggplot, you declare mappings with the `aes()` function inside a `ggplot(...)` call. Let us try the simplest possible call: just the data and the mappings, with nothing drawn yet.

In [ ]:
ggplot(algae_data, aes(x = fertilizer, y = algae, color = pond))

Look carefully at what you just got: **a completely empty plot with labeled axes.** The x-axis is labeled `fertilizer` and runs from 0 to 100. The y-axis is labeled `algae` and runs from about 0 to 100. But there is nothing drawn on the plot.

This is a revealing picture. It tells us that `ggplot2` already *knows* what variables we plan to visualize and which channels they will use — it has even set up the coordinate range — but it has not been told **what marks to draw**. That is the job of the next component.

> **Predict the output.** Before running the next cell, try to describe the plot it will produce. Where will the points be? What will tell the two ponds apart?

## Component 3: Geometries

A **geom** is the visual mark you draw: a point, a line, a bar, a box. You add a geom to a plot with `+`. The `+` is ggplot's composition operator — you read `ggplot(...) + geom_point()` as "take this plot specification and add a point layer to it."

The key conceptual idea: **the same data + the same aesthetics can be drawn with different geoms to answer different questions.** Changing the geom changes the question you are asking.

In [ ]:
# Same aes as before, now with a point geom drawn on top
ggplot(algae_data, aes(x = fertilizer, y = algae, color = pond)) +
  geom_point()

In [ ]:
# Exact same data and aes, different geom — a linear smoother
ggplot(algae_data, aes(x = fertilizer, y = algae, color = pond)) +
  geom_smooth(method = "lm", formula = y ~ x)

In [ ]:
# And you can add layers — both geoms on the same plot
ggplot(algae_data, aes(x = fertilizer, y = algae, color = pond)) +
  geom_point() +
  geom_smooth(method = "lm", formula = y ~ x)

Three plots, one set of aesthetic mappings. The question shifts with the geom:

- `geom_point` asks: *what do individual observations look like?*
- `geom_smooth` asks: *what is the trend, abstracted away from individual points?*
- Both together ask: *how well does the trend fit the individual observations?*

These are genuinely different scientific questions, and the grammar lets you move between them by changing one component.

## Component 4: Statistics

You may have noticed that `geom_smooth` did not just plot your raw data — it *computed* a regression line and drew that instead. Every geom has an associated **statistical transformation**, or `stat`, that determines what gets drawn:

| Geom | Default stat | What it does to your data |
|------|--------------|---------------------------|
| `geom_point` | `stat_identity` | Nothing — plot the rows as-is |
| `geom_histogram` | `stat_bin` | Bin the data, plot the counts |
| `geom_bar` | `stat_count` | Count occurrences of each level |
| `geom_boxplot` | `stat_boxplot` | Compute quartiles, whiskers, outliers |
| `geom_smooth` | `stat_smooth` | Fit a regression, return fitted + CI |

This is easy to miss but conceptually important: **some plots show you your data, and some plots show you a summary of your data.** A histogram, for example, shows none of your original values — it shows counts of bins. When you make a histogram you are doing statistics, not just drawing.

In [ ]:
# A histogram of fertilizer values. The y-axis (count) is computed, not read
# from a column — that is the stat layer at work.
ggplot(algae_data, aes(x = fertilizer)) +
  geom_histogram(bins = 20)

## Component 5: Scales

A **scale** controls the mapping from data values to visual values: what range the x-axis covers, what colors represent what groups, whether the axis is linear or log, and so on. You can usually ignore scales — ggplot picks sensible defaults — but sometimes the convention of a field *requires* a specific scale.

A textbook case in linguistics is the **vowel space**. Phoneticians plot F1 on the y-axis and F2 on the x-axis, but with **both axes reversed**, so that high vowels sit at the top and front vowels sit on the left, matching the anatomy of the mouth. That is a pure scale decision — the data has not changed, only the mapping from data values to pixels.

Let us load a vowel dataset and see.

In [ ]:
raw_lines <- readLines("../data/vowels.csv", encoding = "UTF-8", warn = FALSE)
vowels <- read.csv(textConnection(paste(raw_lines, collapse = "\n")))
head(vowels)

In [ ]:
# First attempt: just F1 vs F2, no scale adjustments. This will look "wrong"
# to a phonetician — high vowels are at the bottom, front vowels on the right.
ggplot(vowels, aes(x = F2, y = F1, color = VOWEL)) +
  geom_point(size = 2)

In [ ]:
# Now fix it with scale layers. Nothing about the data changed — only the
# mapping from data values to pixels.
ggplot(vowels, aes(x = F2, y = F1, color = VOWEL)) +
  geom_point(size = 2) +
  scale_x_reverse() +
  scale_y_reverse()

## Component 6: Coordinates

The **coordinate system** determines how the scaled data values are finally placed on the page. Cartesian is the default and what you almost always want. But there are others:

- `coord_flip()` — swap x and y. Useful for horizontal bar charts when category labels are long.
- `coord_polar()` — turn a bar chart into a pie chart or a radial plot. (Use sparingly; angle is a poor perceptual channel.)
- `coord_fixed()` — force a 1:1 aspect ratio, important when x and y are in the same units.

The distinction between *scales* and *coordinates* is subtle but worth pinning down. A scale changes how a single variable is mapped to a channel (e.g., log the x-axis). A coordinate system changes how all the channels are placed on the page (e.g., treat y as an angle).

## Component 7: Facets

**Faceting** — also known as *small multiples* — repeats the same plot across subsets of the data. This is one of the most powerful ideas in data visualization, and it deserves more attention than it usually gets. Rather than trying to cram four variables into one plot via color and shape and size, you can hold three constant and let the fourth select which panel you are looking at.


In [ ]:
# One panel per vowel, all sharing the same axes — easy to compare shapes
ggplot(vowels, aes(x = F2, y = F1, color = VOWEL)) +
  geom_point(size = 1) +
  scale_x_reverse() +
  scale_y_reverse() +
  facet_wrap(~ VOWEL) +
  labs(title = "Vowel space, faceted by vowel category")

## A perception sidebar: not all channels are equal

You now know the mechanics of the grammar. But the grammar will happily let you make terrible plots. Which choices are *good* ones is a separate question, and it has an empirical answer.

In a classic 1984 experiment, Cleveland and McGill asked people to read values off of different kinds of graphs and measured how accurate they were. The result was a ranking of visual channels by **decoding accuracy** — roughly, how well a reader can recover a numeric value from that channel:

1. **Position along a common scale** (what scatter plots use) — best
2. **Position along non-aligned scales** (small multiples)
3. **Length** (bar charts)
4. **Angle / slope** (pie charts, line charts)
5. **Area** (bubble charts)
6. **Volume, color saturation** — worse
7. **Color hue** (nominal only) — worst for quantity

Two practical rules follow from this ranking:

- **Map your most important variable to position.** If you care about it most, put it on x or y.
- **Reserve color hue for categorical variables.** Hue is great at distinguishing *which group* a point belongs to, and terrible at communicating *how much*. If you see a rainbow gradient encoding a quantity, something has gone wrong.

## Same data, different question

To close the loop on the conceptual point: the grammar lets you re-ask a question of the same data by changing one component. We have already been plotting F1 against F2 — but suppose the question is not "where does each vowel sit in the formant space" but "how much does F1 vary within each vowel category." That is a different question, and it calls for a different geom.

In [ ]:
ggplot(vowels, aes(x = VOWEL, y = F1, color = VOWEL)) +
  geom_boxplot() +
  scale_y_reverse() +
  labs(
    title = "Distribution of F1 within each vowel category",
    x = "Vowel",
    y = "F1 (Hz, reversed)"
  )

A boxplot is one choice from a whole **family** of distribution-summary geoms, all built into ggplot2. Each one makes a different tradeoff between how much of the distribution it shows and how readable the result is:

| Geom | What it shows | Best for |
|------|---------------|----------|
| `geom_boxplot` | Median, quartiles, whiskers, outliers | Quick summaries; many groups |
| `geom_violin` | Smoothed density of the full distribution | Seeing shape — skew, bimodality |
| `geom_density` | 1D kernel density (no grouping variable on x) | Comparing a few distributions overlaid |
| `geom_jitter` | Raw points, with a bit of horizontal noise | Small-*n* data where every observation matters |

The violin plot is worth meeting in particular, because it catches things a boxplot hides. A boxplot summarizes a distribution down to five numbers — that's efficient, but it cannot tell you whether your data is unimodal, bimodal, or has a long tail. A violin shows the whole shape. And because geoms layer, you can combine them.

In [ ]:
ggplot(vowels, aes(x = VOWEL, y = F1, color = VOWEL)) +
  geom_violin() +
  geom_jitter(width = 0.15, size = 0.5, alpha = 0.3) +
  scale_y_reverse() +
  labs(
    title = "F1 distribution per vowel — violin + jittered raw points",
    x = "Vowel",
    y = "F1 (Hz, reversed)"
  )

Notice what changed and what did not. The data is the same. The aesthetic mapping is different — `VOWEL` moved from the color channel (as a secondary distinguisher) to the x channel (as the primary organizing variable). The geom changed from points to boxes, which means the statistic changed from identity to a five-number summary. One grammatical rewrite, one new question answered.

## Critique exercises

For each of the following plotting choices, identify what is **grammatically** wrong — that is, which component has been misused — and what you would change. Don't worry about the exact ggplot function names; just name the component and the fix.

1. A researcher plots F1 (a continuous variable in Hz) against speaker ID, with speaker ID mapped to the x-axis and F1 mapped to y. Speaker IDs are arbitrary strings. The researcher draws a `geom_line()` connecting all the points in order.

2. A study of word frequency maps the log-frequency of each word to **point size** and part of speech to **color**. There are 10,000 words on the plot.

3. A reaction-time study plots a bar chart of mean RT for five conditions, with **the five conditions mapped to five different hues of a rainbow palette**, and no axis labels — the legend tells you which color is which condition.

4. A dialectology paper plots formant data for 12 vowels in 12 different dialects as a **single** scatter plot with `color = dialect` and `shape = vowel`. 144 combinations, one plot.

5. A histogram of vowel duration with **three bins**.

<details>
<summary>Click for suggested answers</summary>

1. **Geom misuse.** `geom_line` implies order and continuity between adjacent x values, but speaker ID is nominal — adjacency is meaningless. Use `geom_point`, or if a summary is wanted, `geom_boxplot` with speaker on x.

2. **Aesthetic channel mismatch.** Size is a poor channel for a continuous quantity (area is hard to decode), and 10,000 points is too many regardless. Position is a better channel for log-frequency (map it to y). The 10,000-point overplotting is a density problem — consider a 2D histogram or a facet by POS.

3. **Scale/perception.** Rainbow palettes suggest ordering where there is none, and forcing the reader to do a legend lookup for every bar is unnecessary when the condition could simply be on the x-axis. Put the condition on x, drop the color mapping entirely (or use a single neutral fill).

4. **Facets, not channels.** You cannot ask color *and* shape to carry 144 distinctions. This is what `facet_wrap(~ dialect)` is for — 12 small multiples of vowel scatter plots, one per dialect.

5. **Statistic parameter.** Three bins hides almost all the structure in the distribution. The bin count is a parameter of the `stat_bin` transformation, and it has to be chosen to reveal the shape — try 20–50 and see what the data supports.

</details>

## Quick syntax reference

You'll look this up — and an LLM will happily write it for you. What matters is that you know which **component** you are reaching for.

| Component | Common functions |
|-----------|------------------|
| Data + mappings | `ggplot(data, aes(x = ..., y = ..., color = ..., shape = ..., size = ...))` |
| Geoms | `geom_point`, `geom_line`, `geom_bar`, `geom_histogram`, `geom_boxplot`, `geom_smooth`, `geom_violin`, `geom_density` |
| Stats (usually implicit) | `stat_bin`, `stat_smooth`, `stat_summary` |
| Scales | `scale_x_log10`, `scale_x_reverse`, `scale_y_continuous`, `scale_color_brewer`, `scale_color_manual` |
| Coordinates | `coord_cartesian`, `coord_flip`, `coord_polar`, `coord_fixed` |
| Facets | `facet_wrap(~ var)`, `facet_grid(rows ~ cols)` |
| Labels + theme | `labs(title=..., x=..., y=...)`, `theme_minimal()`, `theme(...)` |

The composition operator is always `+`, and the order of layers controls draw order (later layers go on top).

## Takeaways

- A plot is a **composition of components**, not a chart type from a menu.
- The most important conceptual move is the **aesthetic mapping**: variables map to visual channels, and the same data supports many plots.
- **Position is the best visual channel**; color hue is for nominal categories; size is for rough comparison only.
- When a plot gets too crowded, the usual fix is to **move a variable out of an aesthetic and into a facet**.
- Deciding *which component you need* is the part of the job that doesn't get outsourced — the syntax does.